In [1]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [2]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [3]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(temperature=0,model = "llama3-70b-8192")

# Create a basic RAG without memory

In [5]:
from langchain.chains import create_retrieval_chain
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.document_loaders import TextLoader
from langchain.schema import HumanMessage, SystemMessage

In [9]:
loader = TextLoader("data/be-good.txt")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vector_db = Chroma.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())
retriever = vector_db.as_retriever() 

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([                                    
    ("system", system_prompt),
    ("human", "{input}"),
])

In [ ]:
question_answer_chain = create_stuff_documents_chain(llmModel,prompt)

rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [11]:
output = rag_chain.invoke({"input": "What is this article about?"})

In [12]:
output

{'input': 'What is this article about?',
 'context': [Document(id='b65f2bb8-8b24-43ba-b0c2-8e46ff2b4d5c', metadata={'source': 'data/be-good.txt'}, page_content='"new economy" during the Bubble.  Believe me, I was not drinking\nthat kool-aid at the time.  But I\'m convinced there were some \ngood\nideas buried in Bubble thinking.  For example, it\'s ok to focus on\ngrowth instead of profitsâ€”but only if the growth is genuine.\nYou can\'t be buying users; that\'s a pyramid scheme.   But a company\nwith rapid, genuine growth is valuable, and eventually markets learn\nhow to value valuable things.[2] The idea of starting\na company with benevolent aims is currently undervalued, because\nthe kind of people who currently make that their explicit goal don\'t\nusually do a very good job.It\'s one of the standard career paths of trustafarians to start\nsome vaguely benevolent business.  The problem with most of them\nis that they either have a bogus political agenda or are feebly\nexecuted.  T

In [13]:
output['answer']

'This article appears to be about the author\'s reflections on the "Bubble" economy and the ideas that were buried within it, as well as the author\'s experiences with startups and entrepreneurship.'

# Step 2: Create a ChatPromptTemplate able to contextualize inputs

In [14]:
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create a retriever aware of this memory

In [16]:
from langchain.chains import create_history_aware_retriever

history_aware_retriever = create_history_aware_retriever(llmModel, retriever, contextualize_q_prompt)

# Create a basic Conversational RAG

In [17]:
qa_prompt=ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])

qa_chain=create_stuff_documents_chain(llmModel, qa_prompt)

rag_chain=create_retrieval_chain(history_aware_retriever, qa_chain)

# Trying the conversational RAG

In [18]:
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []

question = "What is this article about?"

ai_msg_1 = rag_chain.invoke({"input": question, "chat_history": chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=ai_msg_1["answer"]),
    ]
)

second_question = "What was my previous question about?"

ai_msg_2 = rag_chain.invoke({"input": second_question, "chat_history": chat_history})

print(ai_msg_2["answer"])

Your previous question was "What is this article about?"


# Advanced conversational RAG with persistence and session memories

In [19]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

chat_history = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in chat_history:
        chat_history[session_id]=ChatMessageHistory()
    else:
        return chat_history[session_id]
    

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [20]:
session1 = {"configurable":{"session_id":"001"}}

In [22]:
conversational_rag_chain.invoke({"input":"What is this article about?"},
                                config=session1,)["answer"]

'This article appears to be about the author\'s reflections on the "Bubble" economy and the ideas that were buried within it, as well as the author\'s experiences with startups and entrepreneurship.'

In [23]:
conversational_rag_chain.invoke({"input":"What was my previous question about?"},
                                config=session1,)["answer"]

'Your previous question was "What is this article about?"'